# 강의 02 · 실습 2 — 구조화 출력 · (6) 고난도 III

## 1. 문제상황

- 구립 도서관은 하루에 들어온 민원 메일을 저녁에 한 번 모아 담당자에게 넘깁니다.
- 메일마다 보낸 사람, 민원의 종류(시설·도서·이용), 요지, 급한 정도가 문장 속에 섞여 있고, 종류는 보낸 사람이 정하지 않으므로 담당자가 읽고 판단합니다.
- 담당자는 민원을 종류별로 세어 관장에게 보고하고, 급한 민원부터 처리합니다. 급한 정도는 당장 처리 1, 이번 주 2, 여유 3의 세 단계로 관리합니다.
- 모델에게 묶음을 통째로 넘겨 정리시켜 봤더니, 어떤 날은 세 통 중 두 통만 정리해 오고, 종류를 「시설물」「도서관 이용」처럼 정해진 세 가지 밖의 말로 적어 와서 집계표의 칸이 맞지 않았습니다.
- 빠진 민원은 아무 오류도 내지 않고 사라집니다. 사라진 것을 아는 방법이 없다는 것이 가장 큰 문제입니다.

## 2. 문제와 목표

- **문제**: 민원 묶음을 정리한 답에서 항목이 빠져도 아무 오류가 나지 않고, 종류가 정해진 값 밖으로 벗어나도 모양 검사만으로는 걸리지 않습니다.
- **목표**: 민원 한 통의 모양(보낸 사람·종류·요지·급한 정도)과 묶음의 모양(민원 리스트)을 중첩해 선언하고, 종류가 시설·도서·이용 밖이거나 급한 정도가 1·2·3 밖이면 검증에서 걸리게 합니다. 급한 정도는 당장 처리 1, 이번 주 2, 여유 3으로 받습니다. 프로그램이 입력 메일 수를 직접 세어 답의 민원 수와 대조하고, 검증이나 대조에 실패하면 실패 이유를 붙여 다시 호출하는 처리를 둡니다. 다시 호출하는 횟수에는 상한을 둡니다. 통과한 묶음에서 종류별 건수 집계와 급한 민원 목록을 만듭니다.
    - 민원 한 통: 보낸 사람, 종류, 요지, 급한 정도.
    - 종류의 허용 값: 시설·도서·이용.
    - 급한 정도: 1 = 당장 처리, 2 = 이번 주, 3 = 여유.
    - 다시 호출하는 횟수의 상한: 2회. 메일 세 통은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**: 세 통의 민원 메일을 넣었을 때 민원 세 개가 든 묶음 객체가 돌아오고, 종류별 집계의 합이 3이며, 검증 실패 관찰에서 세 값 밖의 종류가 든 답과 민원이 두 개뿐인 답이 각각 걸리는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

## 5. 코드 골격

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import json
import os
import re
from datetime import date

from dotenv import load_dotenv, find_dotenv
from litellm import completion
from pydantic import BaseModel, ValidationError, field_validator

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

MODEL = "openai/gpt-5.6-luna"
# 같은 OPENAI 키로 호출되는 대체 모델(2026-09-05 확인): openai/gpt-4o-mini · openai/gpt-4.1-mini · openai/gpt-5-mini · openai/gpt-5.4-mini
print("준비를 마쳤습니다.")

# 주어진 자료 — 값과 이름을 그대로 씁니다.
MAILBOX = """[메일 1] 보낸 사람: 김하늘
3층 열람실 창가 자리 콘센트가 어제부터 전기가 안 들어옵니다. 노트북 쓰는 사람이 많아서 빨리 봐 주세요.

[메일 2] 보낸 사람: 박준서
예약한 「데이터 분석 입문」이 도착했다는 문자를 받았는데 대출대에서 없다고 합니다. 이번 주 안에 확인 부탁드립니다.

[메일 3] 보낸 사람: 이서연
주말 개관 시간을 한 시간 늘려 주실 수 있는지 건의드립니다. 급한 일은 아닙니다."""


### 단계 ① — 스키마 선언 (요구사항 1, 2)

- 이 단계의 코드를 「4. 단계별 요구사항」에 적은 요구사항과 「5. 코드 골격」에 세운 골격에 맞춰 작성합니다.

In [ ]:
# 여기에 단계 ①(스키마 선언과 검증기)을 작성합니다.

### 단계 ② — 스키마를 건 호출 (요구사항 3)

- 이 단계의 코드를 작성합니다. 호출 함수는 다시 호출할 때 덧붙일 문장을 받을 수 있어야 합니다.
- 입력은 「6. 코드 — 스텝바이스텝」 단계 0에 주어진 메일 세 통입니다. 메일 표지 `[메일 N]`이 접수 건수를 세는 기준입니다.



In [ ]:
# 여기에 단계 ②(추출 규칙, 입력 문서, 스키마를 지정한 호출 함수)를 작성합니다.

### 단계 ③ — 객체 수신·파싱 (요구사항 4)

- 이 단계의 코드를 작성합니다. 검증 실패 뒤의 처리와 다음 단계가 여기에 들어갑니다.

In [ ]:
# 여기에 단계 ③(파싱과 다음 단계)을 작성합니다.

### 단계 ④ — 검증 실패 관찰 (요구사항 5)

- 모델을 부르지 않고 어긋난 입력만으로 검증의 경계를 확인합니다.

In [ ]:
# 여기에 단계 ④(어긋난 입력으로 검증 실패 관찰)를 작성합니다.

## 7. 실행 결과 확인

1. 실행 기록에 통과가 찍히고, 민원 수가 3입니다. 첫 답이 걸렸다면 그 앞에 실패 이유가 찍힙니다.
2. 종류별 집계의 값을 더하면 3이고, 종류 이름은 시설·도서·이용 밖으로 나가지 않습니다. 급한 민원 목록에 급한 정도가 1인 민원만 들어 있습니다.
3. 검증 실패 관찰에서 「시설물」은 종류 필드에서 걸리고, 민원이 두 개뿐인 답은 기대 건수 3과 민원 수 2 두 숫자가 든 오류로 걸립니다.

확인 항목이 모두 맞으면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다.